# Building my Liked Songs dataset

This notebook takes the raw files pulled by the scripts in this folder and turns them into one clean table, `liked_songs_clean.csv`, with one row per song.

| File | Made by | What's in it |
|---|---|---|
| `liked_songs_raw.csv` | `pull_songs.py` | My Spotify Liked Songs: title, artist, album, release date, when I liked it |
| `audio_features.csv` | `pull_audio_features.py` | Danceability, energy, valence, tempo, etc. from ReccoBeats (about 67% of songs) |
| `lastfm_tracks.csv` | `pull_lastfm.py` | Last.fm listeners, playcount, tags, and my own play count per song |
| `lastfm_artists.csv` | `pull_lastfm.py` | Last.fm listeners, playcount, and tags per artist |

Steps: load and clean, merge, remove duplicate saves, then build features in groups (dates, artist/album, genre and tags, popularity, mood, audio, title text, my plays).

In [1]:
import numpy as np
import pandas as pd

# Spotify gives "added_at" in UTC. Converting to my time zone makes "hour liked" mean my local hour.
LOCAL_TZ = "America/New_York"
# The day the data was pulled, used for "days since liked" so the numbers don't change every time this runs
PULL_DATE = pd.Timestamp("2026-10-20", tz=LOCAL_TZ)
# I joined Last.fm on this date, so my Last.fm play counts only cover listening after it
LASTFM_START = pd.Timestamp("2024-12-04", tz=LOCAL_TZ)

pd.set_option("display.max_columns", 60, "display.width", 200)

## 1. Load the raw songs

Two columns are dropped right away:
- `popularity` is empty for every song, because Spotify stopped returning it for new developer apps in 2026. Popularity comes from Last.fm instead.
- `is_local` is only true for the one "ghost" row (no title, no ID, 0 seconds long), which is removed here.

In [2]:
raw = pd.read_csv("liked_songs_raw.csv")
print(f"Raw rows: {len(raw)}")

songs = raw.dropna(subset=["track", "track_id"]).copy()
assert not songs["is_local"].any() and songs["popularity"].isna().all()
songs = songs.drop(columns=["popularity", "is_local"])
print(f"After dropping the empty row: {len(songs)}")

Raw rows: 5261
After dropping the empty row: 5260


## 2. Merge in audio features and Last.fm data

All merges are "left" merges, so every song stays even if it has no audio features or Last.fm match. After merging, the row count is checked to make sure no song got duplicated.

In [3]:
audio = pd.read_csv("audio_features.csv").drop(columns=["reccobeats_id"]).drop_duplicates("track_id")
lfm_tracks = pd.read_csv("lastfm_tracks.csv").drop_duplicates("track_id")
lfm_artists = pd.read_csv("lastfm_artists.csv").drop_duplicates("artist_id")

# Every song has "loved" = False (I don't use the heart button on Last.fm), so it adds nothing
lfm_tracks = lfm_tracks.drop(columns=["lfm_user_loved"])

df = (songs.merge(audio, on="track_id", how="left")
           .merge(lfm_tracks, on="track_id", how="left")
           .merge(lfm_artists, on="artist_id", how="left"))
assert len(df) == len(songs), "a merge created extra rows"

print(f"Rows: {len(df)}")
print(f"With audio features:    {df['energy'].notna().mean():.1%}")
print(f"Matched on Last.fm:     {(df['lfm_match'] != 'none').mean():.1%}")
print(f"Artist found on Last.fm: {df['lfm_artist_matched'].eq(True).mean():.1%}")

Rows: 5260
With audio features:    67.2%
Matched on Last.fm:     100.0%
Artist found on Last.fm: 100.0%


## 3. Remove duplicate saves

Some songs are in my Liked Songs more than once, usually because I liked both the single and the album version. These are the same song to me, so I keep one row per title + artist and count how many times it was saved.

- Titles and artists are compared after lowercasing and removing punctuation, so "Don't" and "Don’t" count as the same.
- The earliest save is kept, since that's when I first liked the song.
- If the earliest version is missing something (like audio features) and another version has it, the gap is filled from the other version.

In [4]:
def normalize(s):
    s = s.str.lower().str.replace("’", "'")
    return s.str.replace(r"[^a-z0-9 ]", "", regex=True).str.split().str.join(" ")

df["added_at"] = pd.to_datetime(df["added_at"], utc=True).dt.tz_convert(LOCAL_TZ)
df = df.sort_values("added_at")
df["_key"] = normalize(df["track"]) + " | " + normalize(df["artist"])
df["times_saved"] = df.groupby("_key")["track_id"].transform("size")

before = len(df)
# first() takes the first non-blank value in each column, in added_at order
df = df.groupby("_key", sort=False).first().reset_index(drop=True)
print(f"{before} rows -> {len(df)} unique songs ({before - len(df)} duplicate saves removed)")
print(df["times_saved"].value_counts().sort_index().rename("songs").to_string())

5260 rows -> 4811 unique songs (449 duplicate saves removed)
times_saved
1    4390
2     394
3      26
4       1


## 4. Date features

- When I liked it: year, month, day of week, hour (my local time).
- When it came out: release year and decade. About 150 albums only list a year, so those release dates are treated as January 1 of that year. That makes "song age" a little rough for those songs.
- Song age when liked: years between release and when I liked it. Negative values (liking a song before its official release date, which happens with pre-saves) are set to 0.
- Binge days: days where I liked a lot of songs at once.

In [5]:
df["liked_year"] = df["added_at"].dt.year
df["liked_month"] = df["added_at"].dt.month
df["liked_dow"] = df["added_at"].dt.day_name()
df["liked_hour"] = df["added_at"].dt.hour

release = pd.to_datetime(df["release_date"], format="mixed", errors="coerce")
print(f"Release dates that couldn't be read: {release.isna().sum()}")
df["release_year"] = release.dt.year
df["release_decade"] = (df["release_year"] // 10 * 10).astype("Int64")

liked_naive = df["added_at"].dt.tz_localize(None)
df["song_age_when_liked"] = ((liked_naive - release).dt.days / 365.25).clip(lower=0).round(1)
df["days_since_liked"] = (PULL_DATE - df["added_at"]).dt.days

liked_day = df["added_at"].dt.date
df["songs_liked_that_day"] = liked_day.map(liked_day.value_counts())
per_day = liked_day.value_counts()
print("\nSongs liked per day (days with at least one like):")
print(per_day.describe(percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())

Release dates that couldn't be read: 0

Songs liked per day (days with at least one like):
count    1730.0
mean        2.8
std         3.5
min         1.0
50%         2.0
75%         3.0
90%         6.0
95%         8.0
99%        18.0
max        45.0


In [6]:
# A "binge" is a day in the top 5% of days by number of songs liked
BINGE_MIN = int(per_day.quantile(0.95))
df["binge_day"] = df["songs_liked_that_day"] >= BINGE_MIN
print(f"Binge threshold: {BINGE_MIN}+ songs in one day")
print(f"{(per_day >= BINGE_MIN).sum()} binge days, holding {df['binge_day'].mean():.1%} of my songs")
print("\nBiggest days:")
print(per_day.head(5).to_string())

Binge threshold: 8+ songs in one day
99 binge days, holding 28.1% of my songs

Biggest days:
added_at
2022-05-21    45
2020-11-22    39
2019-04-03    37
2019-10-23    34
2019-01-21    32


## 5. Artist and album features

In [7]:
df["my_songs_by_artist"] = df.groupby("artist_id")["track_id"].transform("size")
df["is_collab"] = df["n_artists"] > 1
df["is_single"] = df["album_type"] == "single"

# Share of an album's tracks that I liked (only meaningful for albums, a single is always 1.0)
liked_from_album = df.groupby("album_id")["track_id"].transform("size")
df["album_share_liked"] = (liked_from_album / df["album_total_tracks"]).clip(upper=1).round(3)

print("Most-liked artists:")
print(df.drop_duplicates("artist_id").nlargest(8, "my_songs_by_artist")[["artist", "my_songs_by_artist"]].to_string(index=False))
print(f"\nCollabs: {df['is_collab'].mean():.1%}   Singles: {df['is_single'].mean():.1%}")

Most-liked artists:


        artist  my_songs_by_artist
  Taylor Swift                 278
 One Direction                  91
 Morgan Wallen                  75
 Justin Bieber                  72
 Ariana Grande                  65
Chelsea Cutler                  63
    John Mayer                  57
    Kanye West                  51

Collabs: 24.0%   Singles: 28.4%


## 6. Tags and main genre

Spotify no longer shares genres with new developer apps, so genres come from Last.fm tags. Tags are words that Last.fm users attach to songs and artists ("indie pop", "chill", "seen live").

- Only about 28% of songs have their own tags, but about 96% of artists do. So each song uses its own tags when it has them and its artist's tags otherwise. `tag_source` records which one was used.
- **Main genre**: tags are checked in order of how many people used them, and the first one that fits a bucket wins. Tags that aren't genres ("female vocalists", "american", "2019") are skipped.
- Bucket rules are checked top to bottom, so "indie folk" lands in Folk, "pop rock" in Rock, and "alternative rnb" in R&B.

In [8]:
# "hip hop" and "hip-hop" are the same tag spelled two ways
for col in ["lfm_track_tags", "lfm_artist_tags"]:
    df[col] = df[col].str.replace("hip hop", "hip-hop")
df["tags"] = df["lfm_track_tags"].fillna(df["lfm_artist_tags"])
df["tag_source"] = np.select([df["lfm_track_tags"].notna(), df["lfm_artist_tags"].notna()],
                             ["track", "artist"], "none")
print(df["tag_source"].value_counts().to_string())

# (bucket, words that put a tag in it), checked in this order
GENRE_RULES = [
    ("Soundtrack/Musical", ["soundtrack", "musical", "disney", "broadway", "score"]),
    ("Christian", ["christian", "worship", "gospel"]),
    ("Hip-Hop/Rap", ["hip-hop", "hip hop", "rap", "trap", "drill"]),
    ("R&B/Soul", ["rnb", "r&b", "soul", "funk", "motown"]),
    ("Country", ["country", "americana", "bluegrass"]),
    ("Folk/Acoustic", ["folk", "singer-songwriter", "acoustic"]),
    ("Electronic/Dance", ["electronic", "electro", "house", "edm", "techno", "dance", "dubstep",
                          "trance", "synth", "garage", "ambient", "downtempo", "trip hop", "lo-fi", "disco"]),
    ("Indie/Alternative", ["indie", "alternative", "shoegaze", "dream pop", "emo", "grunge"]),
    ("Rock", ["rock", "metal", "punk"]),
    ("Jazz/Blues", ["jazz", "blues", "swing"]),
    ("Pop", ["pop", "boyband", "boy band", "x factor"]),
]


def tag_genre(tag):
    for bucket, words in GENRE_RULES:
        if any(w in tag for w in words):
            return bucket
    return None


def song_genres(track_tags, artist_tags):
    # Song's own tags first, then the artist's (missing tags are NaN, which isn't a string)
    tags = [t for s in (track_tags, artist_tags) if isinstance(s, str) for t in s.split("|")]
    return [g for g in map(tag_genre, tags) if g]


genres = pd.Series([song_genres(t, a) for t, a in zip(df["lfm_track_tags"], df["lfm_artist_tags"])],
                   index=df.index)
df["main_genre"] = genres.map(lambda g: g[0] if g else "Unknown")
df["genre_count"] = genres.map(lambda g: len(set(g)))

print("\nMain genre:")
print(df["main_genre"].value_counts().to_string())

tag_source
artist    3496
track     1258
none        57



Main genre:
main_genre
Pop                   1325
Hip-Hop/Rap            657
Country                548
Indie/Alternative      531
R&B/Soul               420
Electronic/Dance       388
Folk/Acoustic          386
Rock                   302
Unknown                 95
Soundtrack/Musical      92
Jazz/Blues              34
Christian               33


In [9]:
# Check: the most common tags that didn't match any bucket. If a real genre shows up here, add it to GENRE_RULES.
all_tags = df["tags"].dropna().str.split("|").explode()
print(all_tags[all_tags.map(tag_genre).isna()].value_counts().head(20).to_string())

tags
female vocalists        654
american                455
british                 383
male vocalists          299
usa                     197
my top songs            172
canadian                 97
one direction            87
guilty pleasure          78
70s                      75
love                     74
justin bieber            70
deezer                   63
love at first listen     56
60s                      54
want to see live         54
80s                      52
chicago                  51
australian               49
oldies                   44


In [10]:
# Yes/no flag for each of the 15 most common tags, e.g. tag_indie_pop
top_tags = all_tags.value_counts().head(15).index
for t in top_tags:
    col = "tag_" + t.replace(" ", "_").replace("-", "_")
    df[col] = df["tags"].fillna("").str.split("|").map(lambda tags: t in tags)
print("Tag flags added:", ", ".join(top_tags))

Tag flags added: pop, hip-hop, indie, rnb, rock, female vocalists, singer-songwriter, indie pop, country, rap, electronic, folk, american, alternative, british


## 7. Popularity (from Last.fm)

Spotify's popularity score, artist popularity, and follower counts are no longer available, so these use Last.fm listener and play counts instead. Those numbers are very skewed (a few songs have millions of listeners, most have far fewer), so they're put on a log10 scale: 3 = a thousand, 6 = a million.

- `popularity_tier`: low / mid / high, splitting my songs into thirds by listeners.
- `niche_score`: song listeners vs. the artist's listeners, on the log scale. Near 0 means this song is about as big as the artist (one of their hits). Very negative means a deep cut.
- `mainstream_score`: 0 to 100, my songs ranked by the average of listeners and playcount (both log scale, standardized). Under 20 = `hidden_gem`.
- `replays_per_listener`: playcount divided by listeners, a rough "do people keep coming back to it" measure.
- `artist_size`: bucketed artist listeners.

In [11]:
df["log_track_listeners"] = np.log10(df["lfm_track_listeners"] + 1)
df["log_track_playcount"] = np.log10(df["lfm_track_playcount"] + 1)
log_artist = np.log10(df["lfm_artist_listeners"] + 1)

df["popularity_tier"] = pd.qcut(df["log_track_listeners"], 3, labels=["low", "mid", "high"])
df["niche_score"] = (df["log_track_listeners"] - log_artist).round(3)

z = lambda s: (s - s.mean()) / s.std()
combined = (z(df["log_track_listeners"]) + z(df["log_track_playcount"])) / 2
df["mainstream_score"] = (combined.rank(pct=True) * 100).round(1)
df["hidden_gem"] = df["mainstream_score"] < 20

df["replays_per_listener"] = (df["lfm_track_playcount"] / df["lfm_track_listeners"]).round(2)
df["artist_size"] = pd.cut(df["lfm_artist_listeners"], [0, 1e5, 1e6, 3e6, np.inf],
                           labels=["under 100K", "100K-1M", "1M-3M", "3M+"], include_lowest=True)

print(df.groupby("popularity_tier", observed=True)["lfm_track_listeners"].agg(["min", "median", "max"]).astype(int).to_string())
print()
print(df["artist_size"].value_counts().sort_index().to_string())
print("\nBiggest hidden gems (fewest Last.fm listeners):")
print(df.nsmallest(5, "mainstream_score")[["track", "artist", "lfm_track_listeners"]].to_string(index=False))

                    min  median      max
popularity_tier                         
low                   2   32940    92983
mid               92987  211260   424832
high             425387  805968  3842251

artist_size
under 100K     226
100K-1M       1381
1M-3M         1523
3M+           1681

Biggest hidden gems (fewest Last.fm listeners):
                                           track                 artist  lfm_track_listeners
Who Needs Friends? (When You Don't Get Invited?)            Elisia Song                  2.0
            Half the World Away in One Direction           Colsen Brown                  4.0
                                    Jingle Bells          Todd Chrisley                 20.0
     Love's in the Melody (feat. Caleb Keolanui)            Kimié Miner                 36.0
                                           Shout RJ Reynolds a Cappella                  9.0


## 8. Mood

Mood comes from two places:
- **Tags**: only a few hundred songs have mood words like "chill" or "sad", so this covers few songs. The bare tag "love" is left out because on Last.fm it usually means "I love this song", not "love song".
- **Audio**: valence (how positive it sounds) and energy, each 0 to 1, split at 0.5 into four quadrants. This covers the songs with audio features.

The final `mood` uses the audio quadrant when there is one and the tag otherwise. Audio comes first because it's measured the same way for every song, while a tag can come from just a few listeners (the two agree only about a third of the time, see below). `mood_source` says which one was used.

In [12]:
MOOD_TAGS = {
    "calm": {"chill", "chillout", "mellow", "relax", "relaxing", "calm", "dreamy"},
    "sad": {"sad", "melancholic", "melancholy", "heartbreak", "emotional"},
    "upbeat": {"happy", "upbeat", "fun", "feel good", "summer", "party"},
    "romantic": {"love songs", "romantic", "sexy"},
}


def tag_mood(tags):
    for t in str(tags).split("|"):
        for mood, words in MOOD_TAGS.items():
            if t in words:
                return mood
    return None


df["mood_from_tags"] = df["tags"].map(tag_mood)

has_audio = df["valence"].notna() & df["energy"].notna()
quadrant = np.select(
    [(df["valence"] >= .5) & (df["energy"] >= .5), (df["valence"] < .5) & (df["energy"] >= .5),
     (df["valence"] < .5) & (df["energy"] < .5)],
    ["upbeat", "intense", "sad"], "calm")
df["mood_quadrant"] = pd.Series(quadrant, index=df.index).where(has_audio)

df["mood"] = df["mood_quadrant"].fillna(df["mood_from_tags"])
df["mood_source"] = np.select([has_audio, df["mood_from_tags"].notna()], ["audio", "tags"], "none")

print(df["mood_source"].value_counts().to_string())
print()
print(df["mood"].value_counts(dropna=False).to_string())
# How often do tags and audio agree when a song has both?
both = df["mood_from_tags"].notna() & has_audio
print(f"\nTags and audio agree on {(df.loc[both, 'mood_from_tags'] == df.loc[both, 'mood_quadrant']).mean():.0%} of the {both.sum()} songs with both")

mood_source
audio    3407
none     1343
tags       61

mood
NaN         1343
upbeat      1320
intense     1120
sad          781
calm         235
romantic      12

Tags and audio agree on 33% of the 104 songs with both


## 9. Audio features

Only for the ~67% of songs ReccoBeats knew. These are ReccoBeats' own estimates, not Spotify's original numbers.

In [13]:
# A tempo of 0 means it couldn't be measured
df["tempo"] = df["tempo"].replace(0, np.nan)
df["tempo_bucket"] = pd.cut(df["tempo"], [0, 90, 120, 140, np.inf], labels=["slow", "moderate", "upbeat", "fast"])
df["party_score"] = ((df["danceability"] + df["energy"]) / 2).round(3)

print(df["tempo_bucket"].value_counts(dropna=False).to_string())
print("\nTop party songs:")
print(df.nlargest(5, "party_score")[["track", "artist", "party_score"]].to_string(index=False))

tempo_bucket
NaN         1405
moderate    1264
fast         831
upbeat       801
slow         510

Top party songs:
            track      artist  party_score
Super Freaky Girl Nicki Minaj        0.920
     Boogie Fever The Sylvers        0.900
        All I Got   Hendersin        0.886
            Apple  Charli xcx        0.881
            Pause     Pitbull        0.877


## 10. Title text

In [14]:
title = df["track"].str.lower()
df["title_length"] = df["track"].str.len()
df["title_words"] = df["track"].str.split().str.len()
df["has_feat"] = title.str.contains(r"\(feat\.|\(ft\.|\(with |\[feat\.| feat\. ", regex=True)
df["is_remix"] = title.str.contains("remix")
df["is_live"] = title.str.contains(r"\blive\b") & title.str.contains(r"\(live|- live|live at|live from|spotify live")

print(df[["has_feat", "is_remix", "is_live"]].sum().to_string())

has_feat    487
is_remix     78
is_live      38


## 11. My own plays (Last.fm)

`lfm_user_playcount` is how many times I've played the song since I joined Last.fm on December 4, 2024 (Last.fm only knows about listening after I connected it). A 0 does **not** mean I never played it, just not since then. Songs I liked after joining are a fairer test, so `liked_after_lastfm` marks them.

In [15]:
df["liked_after_lastfm"] = df["added_at"] >= LASTFM_START
df["played_since_lastfm"] = df["lfm_user_playcount"] > 0

print(df.groupby("liked_after_lastfm")["played_since_lastfm"].mean().rename("share played").round(3).to_string())
print("\nMost played since joining Last.fm:")
print(df.nlargest(8, "lfm_user_playcount")[["track", "artist", "lfm_user_playcount"]].to_string(index=False))

liked_after_lastfm
False    0.845
True     0.901

Most played since joining Last.fm:
                        track         artist  lfm_user_playcount
             The Hardest Part    Olivia Dean               109.0
                 Another Life            SZA               108.0
          Scorsese Baby Daddy            SZA                99.0
                   Hard Drugs          Susto                99.0
Sally, When The Wine Runs Out     ROLE MODEL                96.0
                         Dive    Olivia Dean                81.0
               Some Protector     ROLE MODEL                81.0
            luther (with sza) Kendrick Lamar                80.0


**Replay rate.** Raw play counts favor songs that have had more time to be played. `plays_per_month` divides my plays by the months the song could have been counted: from whichever came later, liking it or joining Last.fm, up to the pull date. Songs with less than a month of that window are left blank, since a few plays in a week would look like a huge rate.

One limitation: Last.fm counts every play since I joined, including plays from *before* I liked the song, so for songs liked after December 2024 this slightly overstates the rate.

In [16]:
counted_from = df["added_at"].where(df["added_at"] > LASTFM_START, LASTFM_START)
months_counted = (PULL_DATE - counted_from).dt.days / 30.44
df["plays_per_month"] = (df["lfm_user_playcount"] / months_counted).where(months_counted >= 1).round(2)

print(df["plays_per_month"].describe(percentiles=[.5, .9, .99]).round(2).to_string())
print("\nHighest replay rate:")
print(df.nlargest(8, "plays_per_month")[["track", "artist", "lfm_user_playcount", "plays_per_month"]].to_string(index=False))

count    4779.00
mean        0.38
std         0.64
min         0.00
50%         0.18
90%         0.98
99%         3.29
max         7.25

Highest replay rate:
                                                                          track           artist  lfm_user_playcount  plays_per_month
                                             Try and Love Again - 2013 Remaster           Eagles                35.0             7.25
                                                               The Hardest Part      Olivia Dean               109.0             6.05
                                                     Let Alone The One You Love      Olivia Dean                75.0             5.88
                                                                Life Ain't Easy       Mac Miller                27.0             5.75
                                                              Never Comin' Back Flatland Cavalry                25.0             5.64
                                      

## 12. Discovery and taste change

- `is_new_artist`: this was my first liked song by that (main) artist. My earliest likes are almost all "new" by definition, so this is most meaningful after the first year.
- `days_release_to_like`: days from the song's release to when I liked it, and `liked_first_week` for songs I liked within 7 days of release. Only for songs with an exact release day (not year-only). Negative values are pre-saves, liked before the official release.
- `novelty`: how different a song sounds from what I'd liked in the 6 months before it. It's the distance between the song's standardized audio features and the average of my likes in that window (0 = sounds exactly like my recent average). Blank when the window has fewer than 20 songs with audio features.

In [17]:
df = df.sort_values("added_at").reset_index(drop=True)

df["is_new_artist"] = df.groupby("artist_id").cumcount() == 0

exact_day = df["release_date_precision"] == "day"
days_to_like = df["added_at"].dt.tz_localize(None) - pd.to_datetime(df["release_date"], format="mixed")
df["days_release_to_like"] = days_to_like.dt.days.where(exact_day)
df["liked_first_week"] = df["days_release_to_like"].between(0, 7)

print("Share of likes that were a new artist, by year:")
print(df.groupby("liked_year")["is_new_artist"].mean().round(2).to_string())
print(f"\nLiked in the first week after release: {df['liked_first_week'].mean():.1%}")
print(f"Pre-saves (liked before release): {(df['days_release_to_like'] < 0).sum()}")

Share of likes that were a new artist, by year:
liked_year
2015    0.42
2016    0.42
2017    0.38
2018    0.35
2019    0.32
2020    0.28
2021    0.25
2022    0.21
2023    0.25
2024    0.20
2025    0.24
2026    0.28

Liked in the first week after release: 12.3%
Pre-saves (liked before release): 5


In [18]:
NOVELTY_FEATURES = ["danceability", "energy", "valence", "acousticness", "speechiness",
                    "instrumentalness", "tempo", "loudness"]
has_audio = df[NOVELTY_FEATURES].notna().all(axis=1)
audio_songs = df[has_audio].set_index("added_at")
z = (audio_songs[NOVELTY_FEATURES] - audio_songs[NOVELTY_FEATURES].mean()) / audio_songs[NOVELTY_FEATURES].std()

# Average sound of everything liked in the 180 days before each song (closed="left" leaves the song itself out)
recent_avg = z.rolling("180D", closed="left").mean()
recent_n = z["energy"].rolling("180D", closed="left").count()
novelty = np.sqrt(((z - recent_avg) ** 2).sum(axis=1)).where(recent_n >= 20)

df.loc[has_audio, "novelty"] = novelty.round(3).values

print(df["novelty"].describe().round(2).to_string())
print("\nMost novel likes (least like what I'd been liking):")
print(df.nlargest(6, "novelty")[["track", "artist", "liked_year", "novelty"]].to_string(index=False))

count    3386.00
mean        2.57
std         1.16
min         0.66
25%         1.88
50%         2.35
75%         2.95
max        12.54

Most novel likes (least like what I'd been liking):
                                                                                     track                             artist  liked_year  novelty
            The Carnival of the Animals, R. 125: XIII. The Swan (Arr. for Cello and Piano)                Camille Saint-Saëns        2020   12.535
                                                                     Nemo Egg (Main Title)                      Thomas Newman        2020   12.363
Saint-Saëns: Le carnaval des animaux: XIII. Le cygne (Arr. Hazell for Cello and Orchestra)                Camille Saint-Saëns        2025   11.517
                                                            4 Quarters In The Black (Skit)                  Chance the Rapper        2019   11.009
                                                                            

## 13. Save

In [19]:
df["added_at"] = df["added_at"].dt.strftime("%Y-%m-%d %H:%M:%S")  # local time, no time zone suffix
df.to_csv("liked_songs_clean.csv", index=False)
print(f"Saved liked_songs_clean.csv: {df.shape[0]} rows, {df.shape[1]} columns")

nulls = df.isna().mean().round(3)
print("\nColumns with missing values (share missing):")
print(nulls[nulls > 0].sort_values(ascending=False).to_string())

Saved liked_songs_clean.csv: 4811 rows, 99 columns

Columns with missing values (share missing):
mood_from_tags          0.966
lfm_track_tags          0.739
novelty                 0.296
energy                  0.292
isrc                    0.292
acousticness            0.292
danceability            0.292
liveness                0.292
key                     0.292
instrumentalness        0.292
loudness                0.292
tempo                   0.292
valence                 0.292
speechiness             0.292
mode                    0.292
party_score             0.292
mood_quadrant           0.292
tempo_bucket            0.292
mood                    0.279
days_release_to_like    0.031
lfm_artist_tags         0.012
tags                    0.012
plays_per_month         0.007
